In [3]:
import pandas as pd
from pathlib import Path

In [4]:
# Root path till din data
DATA_PATH = Path("../data/raw/Participants") 


order_A = [
    1,4,7,10,15,20,25,30,32,36,2,3,5,6,8,9
]

order_B = [
    1,4,7,10,12,14,16,18,23,28,33,2,3,5,6,8,9
  
]


In [5]:
def extract_index(filename):
    try:
        return int(filename.split("_")[0])
    except:
        return None


def extract_condition(filename):
    parts = filename.replace(".csv", "").split("_")
    
    if len(parts) > 3:
        return parts[-1]
    return "default"

In [6]:
def process_participant(participant_id):
    
    participant_path = DATA_PATH / f"Part{participant_id}" / "by_block"
    
    files = list(participant_path.glob("*.csv"))
    
    #Filtrera rätt filer
    files = [f for f in files if "gsr_ppg" in f.name]
    
    file_info = []
    
    for f in files:
        idx = extract_index(f.name)
        if idx is not None:
            file_info.append((f, idx))
    
    # Välj ordning
    if participant_id <= 11:
        order = order_A
    else:
        order = order_B
    
    #  Filtrera endast valda index
    file_info_filtered = [
        (f, idx) for (f, idx) in file_info if idx in order
    ]
    
    # 🔴 Kontroll
    selected_indices = [idx for _, idx in file_info_filtered]
    missing = [i for i in order if i not in selected_indices]
    
    if missing:
        print(f"⚠️ Part{participant_id} saknar index: {missing}")
    
    # 🔴 Sortera enligt din ordning
    order_map = {val: i for i, val in enumerate(order)}
    
    file_info_sorted = sorted(
        file_info_filtered,
        key=lambda x: order_map[x[1]]
    )
    
    print(f"Part{participant_id} ordning:", [idx for _, idx in file_info_sorted])
    
    # 🔴 Läs in data
    dfs = []
    
    for file_path, idx in file_info_sorted:
        df = pd.read_csv(file_path)
        
        # 🔴 Säkerställ kolumner
        expected_cols = ["Timestamp", "PythonTimestamp", "accelx", "accely", "accelz", "ppg", "gsr"]
        df = df[[col for col in expected_cols if col in df.columns]]
        
        # 🔴 Metadata
        df["participant"] = participant_id
        df["block_index"] = idx
        df["condition"] = extract_condition(file_path.name)
        
        # 🔴 SUPER VIKTIG
        df["sample_index"] = range(len(df))
        
        dfs.append(df)
    
    if len(dfs) == 0:
        raise ValueError(f"Inga filer hittades för Part{participant_id}")
    
    participant_df = pd.concat(dfs, ignore_index=True)
    
    return participant_df

In [7]:
df_test = process_participant(49)

df_test.head()

Part49 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]


,Timestamp,PythonTimestamp,accelx,accely,accelz,ppg,gsr,participant,block_index,condition,sample_index
0,2255806,1.553516e+09,1995,2811,1866,1056.410256,344.262295,49,1,,0
1,2255934,1.553516e+09,1996,2812,1866,1051.282051,344.262295,49,1,,1
2,2256062,1.553516e+09,1999,2820,1867,1044.688645,344.088732,49,1,,2
3,2256190,1.553516e+09,1997,2824,1868,1037.362637,344.088732,49,1,,3
4,2256318,1.553516e+09,1997,2827,1869,1031.501832,344.088732,49,1,,4


In [8]:
from pathlib import Path

# 🔴 din befintliga processed-mapp
BASE_OUTPUT = Path("../data/processed")

# 🔴 ny undermapp
OUTPUT_PATH = BASE_OUTPUT / "fix_order"

# skapa mappen om den inte finns
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("Sparar i:", OUTPUT_PATH)

Sparar i: ..\data\processed\fix_order


In [9]:
all_participants = []

for p in range(49, 61):
    print(f"\nProcessing Part{p}")
    
    try:
        df_p = process_participant(p)
        
        # 🔴 spara i fix_order-mappen
        output_file = OUTPUT_PATH / f"participant_{p}.csv"
        df_p.to_csv(output_file, index=False)
        
        all_participants.append(df_p)
        
    except Exception as e:
        print(f"❌ Fel vid Part{p}: {e}")


Processing Part49
Part49 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part50
Part50 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part51
Part51 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part52
Part52 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part53
Part53 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part54
Part54 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part55
Part55 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part56
Part56 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part57
Part57 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part58
Part58 ordning: [1, 4, 7, 10, 12, 14, 16, 18, 23, 28, 33, 2, 3, 5, 6, 8, 9]

Processing Part59
Part59 ordning: [1, 4, 7, 10, 1

In [16]:
master_df = pd.concat(all_participants, ignore_index=True)

master_file = OUTPUT_PATH / "all_participants.csv"
master_df.to_csv(master_file, index=False)

print("✅ Klar! Sparat i:", OUTPUT_PATH)

✅ Klar! Sparat i: ..\data\processed\fix_order
